# Deterministic Policy Gradients
## DDPG and TD3: Off-Policy Continuous Control

This notebook provides a self-contained, from-scratch implementation of **DDPG** (Deep Deterministic Policy Gradient) and **TD3** (Twin Delayed DDPG) for continuous control. We derive the deterministic policy gradient theorem, implement both algorithms with experience replay and target networks, train them on the Pendulum task, and study the improvements TD3 brings over DDPG.

**What you'll learn:**
1. Deterministic policy gradient theorem
2. DDPG: deep deterministic policy gradient with experience replay
3. Ornstein-Uhlenbeck noise for exploration
4. TD3: twin critics, delayed policy updates, target policy smoothing
5. Why TD3 fixes DDPG's overestimation and instability issues
6. Continuous control on Pendulum

**Prerequisites:** DQN (Notebook 6), Actor-Critic (Notebook 8), policy gradients.

**References:**
- Silver et al., *Deterministic Policy Gradient Algorithms*, ICML, 2014 (DPG).
- Lillicrap et al., *Continuous Control with Deep Reinforcement Learning*, ICLR, 2016 (DDPG).
- Fujimoto et al., *Addressing Function Approximation Error in Actor-Critic Methods*, ICML, 2018 (TD3).

---
## 1. Imports and Configuration

In [ ]:
# ============================================================
#  Imports and Configuration
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from collections import deque
import random
import copy
from typing import List, Tuple, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# ---- Reproducibility ----
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# ---- Hyperparameters ----
GAMMA = 0.99              # Discount factor
TAU = 0.005               # Soft target update rate
LR_ACTOR = 1e-3           # Actor learning rate
LR_CRITIC = 1e-3          # Critic learning rate
BATCH_SIZE = 256          # Mini-batch size
BUFFER_SIZE = 100_000     # Replay buffer capacity
POLICY_DELAY = 2          # TD3: update actor every d steps
POLICY_NOISE = 0.2        # TD3: target policy smoothing noise
NOISE_CLIP = 0.5          # TD3: clipping range for target noise
EXPLORATION_NOISE = 0.1   # Gaussian exploration noise std
N_EPISODES = 300          # Training episodes
HIDDEN_DIM = 256          # Hidden layer size
WARMUP_STEPS = 1000       # Random actions before training

# ---- Plot style ----
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})
COLORS = ['steelblue', 'coral', 'seagreen', 'goldenrod', 'mediumpurple']

# ---- Device ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Gymnasium version: {gym.__version__}")

---
## 2. Deterministic Policy Gradient Theorem

In standard policy gradient methods we learn a **stochastic** policy $\pi_\theta(a|s)$ that
outputs a probability distribution over actions. For continuous action spaces, this
typically means parameterising a Gaussian $\mathcal{N}(\mu_\theta(s), \sigma^2)$ and
sampling from it.

Silver et al. (2014) showed that we can instead learn a **deterministic** policy
$\mu_\theta(s)$ that directly outputs a single action, and still compute a valid
policy gradient. The key result is the **Deterministic Policy Gradient (DPG) theorem**:

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{s \sim \rho^\mu}\left[\nabla_\theta \mu_\theta(s) \nabla_a Q^\mu(s,a)\big|_{a=\mu_\theta(s)}\right]}$$

where $\rho^\mu$ is the state visitation distribution under the deterministic policy.

**Key advantages of deterministic policies:**
- No need to integrate over the action space (lower variance)
- The gradient only requires the gradient of Q w.r.t. actions (via chain rule)
- More sample-efficient: no entropy from stochastic sampling

**Challenge:** A deterministic policy does not explore on its own. We must add
external noise for exploration during training.

---
## 3. DDPG: Deep Deterministic Policy Gradient

**DDPG** (Lillicrap et al., 2016) combines the DPG theorem with ideas from DQN:

1. **Actor-Critic architecture**: The actor $\mu_\theta(s)$ outputs deterministic actions,
   the critic $Q_\phi(s, a)$ estimates action values.

2. **Experience replay**: Store transitions $(s, a, r, s', d)$ in a replay buffer and
   sample mini-batches for training (breaks temporal correlations).

3. **Target networks**: Maintain slowly-updated copies $\mu_{\theta'}$ and $Q_{\phi'}$
   for computing TD targets (stabilises training).

4. **Soft target updates**: Instead of periodic hard copies, blend target weights
   every step:
   $$\theta' \leftarrow \tau\theta + (1-\tau)\theta'$$
   with $\tau \ll 1$ (e.g., 0.005).

5. **Ornstein-Uhlenbeck (OU) noise** for temporally correlated exploration:
   $$x_{t+1} = x_t + \theta_{\text{OU}}(\mu_{\text{OU}} - x_t) + \sigma_{\text{OU}} \mathcal{N}(0, 1)$$
   This produces smooth, momentum-like exploration, which can be beneficial for
   physical control tasks.

```
DDPG Update:
  Critic target:  y = r + gamma * Q_phi'(s', mu_theta'(s'))
  Critic loss:    L = E[(Q_phi(s, a) - y)^2]
  Actor loss:     L = -E[Q_phi(s, mu_theta(s))]
  Soft update:    theta' <- tau * theta + (1 - tau) * theta'
```

---
## 4. TD3: Twin Delayed DDPG

Fujimoto et al. (2018) identified that DDPG suffers from **Q-value overestimation**
(similar to DQN) and proposed three targeted fixes, collectively called **TD3**:

### Trick 1: Twin Critics
Maintain two independent critic networks $Q_{\phi_1}$ and $Q_{\phi_2}$. Use the
**minimum** of their target predictions to compute TD targets:

$$\boxed{y = r + \gamma \min\left(Q_{\phi_1'}(s', \tilde{a}'),\; Q_{\phi_2'}(s', \tilde{a}')\right)}$$

This combats overestimation by taking the more pessimistic estimate.

### Trick 2: Delayed Policy Updates
Update the actor (and target networks) only every $d$ steps (e.g., $d=2$),
while updating the critic every step. This allows the critic to become more
accurate before the actor chases its estimates.

### Trick 3: Target Policy Smoothing
Add clipped noise to the target action to prevent the policy from exploiting
narrow peaks in the Q-function:

$$\tilde{a}' = \mu_{\theta'}(s') + \text{clip}(\epsilon, -c, c), \quad \epsilon \sim \mathcal{N}(0, \sigma)$$

This acts as a regulariser, smoothing the Q-function over similar actions.

---
## 5. Replay Buffer

In [ ]:
# ============================================================
#  Replay Buffer for continuous-action off-policy learning
# ============================================================

class ReplayBuffer:
    """Fixed-size replay buffer storing (s, a, r, s', done) transitions.
    
    Unlike the DQN replay buffer, actions here are continuous (float vectors)
    rather than discrete integers.
    """
    
    def __init__(self, capacity: int = BUFFER_SIZE, seed: int = SEED):
        self.buffer = deque(maxlen=capacity)
        random.seed(seed)
    
    def push(self, state: np.ndarray, action: np.ndarray, reward: float,
             next_state: np.ndarray, done: bool):
        """Add a transition to the buffer."""
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size: int = BATCH_SIZE) -> Tuple[torch.Tensor, ...]:
        """Sample a random mini-batch and return as tensors on device."""
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.FloatTensor(np.array(actions)).to(device),
            torch.FloatTensor(np.array(rewards)).unsqueeze(1).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(np.array(dones)).unsqueeze(1).to(device),
        )
    
    def __len__(self) -> int:
        return len(self.buffer)


# Quick test
buf = ReplayBuffer(capacity=100)
for i in range(50):
    buf.push(np.random.randn(3), np.random.randn(1), float(i), np.random.randn(3), False)
s, a, r, ns, d = buf.sample(8)
print(f"Buffer size: {len(buf)}")
print(f"Sample shapes: states={s.shape}, actions={a.shape}, rewards={r.shape}, "
      f"next_states={ns.shape}, dones={d.shape}")

---
## 6. Ornstein-Uhlenbeck Noise

In [ ]:
# ============================================================
#  Ornstein-Uhlenbeck Process for temporally correlated noise
# ============================================================

class OrnsteinUhlenbeckNoise:
    """Ornstein-Uhlenbeck process for generating temporally correlated noise.
    
    The OU process is a mean-reverting stochastic process:
        dx = theta * (mu - x) * dt + sigma * dW
    
    This produces smooth exploration trajectories, which can be helpful
    for physical systems with inertia (e.g., robotic arms, locomotion).
    
    Args:
        size:   dimensionality of the noise (matches action dim)
        mu:     long-term mean (typically 0)
        theta:  mean reversion rate (how fast noise returns to mu)
        sigma:  volatility (noise magnitude)
    """
    
    def __init__(self, size: int, mu: float = 0.0, theta: float = 0.15,
                 sigma: float = 0.2, seed: int = SEED):
        self.size = size
        self.mu = mu * np.ones(size)
        self.theta = theta
        self.sigma = sigma
        self.rng = np.random.RandomState(seed)
        self.reset()
    
    def reset(self):
        """Reset the internal state to the mean."""
        self.state = self.mu.copy()
    
    def __call__(self) -> np.ndarray:
        """Generate one noise sample and update internal state."""
        dx = (self.theta * (self.mu - self.state)
              + self.sigma * self.rng.randn(self.size))
        self.state += dx
        return self.state.copy()


# Demonstrate OU noise vs Gaussian noise
ou = OrnsteinUhlenbeckNoise(size=1, sigma=0.3)
ou_samples = [ou()[0] for _ in range(200)]
gauss_samples = [np.random.randn() * 0.3 for _ in range(200)]

print(f"OU noise: mean={np.mean(ou_samples):.3f}, std={np.std(ou_samples):.3f}")
print(f"Gaussian: mean={np.mean(gauss_samples):.3f}, std={np.std(gauss_samples):.3f}")

---
## 7. Actor Network

In [ ]:
# ============================================================
#  Actor Network: deterministic policy mu_theta(s) -> action
# ============================================================

class Actor(nn.Module):
    """Deterministic actor for continuous control.
    
    Architecture:
        state -> fc1 (ReLU) -> fc2 (ReLU) -> fc3 (tanh) -> scaled action
    
    The output uses tanh to bound actions to [-1, 1], which is then scaled
    to the environment's action range [action_low, action_high].
    """
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = HIDDEN_DIM,
                 max_action: float = 1.0):
        super().__init__()
        self.max_action = max_action
        
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        """Forward pass: state -> deterministic action in [-max_action, max_action]."""
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        # tanh squashes to [-1, 1], then scale to action bounds
        return self.max_action * torch.tanh(self.fc3(x))


# Quick test
test_actor = Actor(state_dim=3, action_dim=1, max_action=2.0)
test_s = torch.randn(4, 3)  # batch of 4 states
test_a = test_actor(test_s)
print(f"Actor output shape: {test_a.shape}")
print(f"Action range: [{test_a.min().item():.3f}, {test_a.max().item():.3f}]")
print(f"Actor params: {sum(p.numel() for p in test_actor.parameters()):,}")

---
## 8. Critic Network

In [ ]:
# ============================================================
#  Critic Network: Q_phi(s, a) -> scalar Q-value
# ============================================================

class Critic(nn.Module):
    """Q-value critic that takes state and action as input.
    
    Architecture:
        [state, action] (concatenated) -> fc1 (ReLU) -> fc2 (ReLU) -> fc3 -> Q-value
    
    Unlike discrete-action DQN which outputs Q(s, a) for all actions,
    this critic takes a specific continuous action as input and outputs
    a single Q-value.
    """
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = HIDDEN_DIM):
        super().__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
    
    def forward(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        """Forward pass: (state, action) -> Q-value scalar."""
        # Concatenate state and action along feature dimension
        x = torch.cat([state, action], dim=-1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


# Quick test
test_critic = Critic(state_dim=3, action_dim=1)
test_q = test_critic(test_s, test_a.detach())
print(f"Critic output shape: {test_q.shape}")
print(f"Q-values: {test_q.detach().squeeze().numpy().round(3)}")
print(f"Critic params: {sum(p.numel() for p in test_critic.parameters()):,}")

---
## 9. Twin Critic Network (for TD3)

In [ ]:
# ============================================================
#  Twin Critic for TD3: two independent Q-networks
# ============================================================

class TwinCritic(nn.Module):
    """Two independent critic networks for TD3's clipped double Q-learning.
    
    Each critic has the same architecture as the single Critic above,
    but they are initialised independently so they learn different
    approximations of Q(s, a). Using min(Q1, Q2) for TD targets
    combats overestimation.
    """
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = HIDDEN_DIM):
        super().__init__()
        # Critic 1
        self.fc1_1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.fc1_2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc1_3 = nn.Linear(hidden_dim, 1)
        
        # Critic 2
        self.fc2_1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.fc2_2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc2_3 = nn.Linear(hidden_dim, 1)
    
    def forward(self, state: torch.Tensor, action: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Forward pass through both critics.
        
        Returns:
            (Q1, Q2): Q-value estimates from both critics
        """
        x = torch.cat([state, action], dim=-1)
        
        # Critic 1 forward
        q1 = F.relu(self.fc1_1(x))
        q1 = F.relu(self.fc1_2(q1))
        q1 = self.fc1_3(q1)
        
        # Critic 2 forward
        q2 = F.relu(self.fc2_1(x))
        q2 = F.relu(self.fc2_2(q2))
        q2 = self.fc2_3(q2)
        
        return q1, q2
    
    def q1(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        """Forward pass through critic 1 only (used for actor updates)."""
        x = torch.cat([state, action], dim=-1)
        q1 = F.relu(self.fc1_1(x))
        q1 = F.relu(self.fc1_2(q1))
        return self.fc1_3(q1)


# Quick test
test_twin = TwinCritic(state_dim=3, action_dim=1)
q1, q2 = test_twin(test_s, test_a.detach())
q1_only = test_twin.q1(test_s, test_a.detach())
print(f"Twin critic Q1 shape: {q1.shape}, Q2 shape: {q2.shape}")
print(f"Q1 values: {q1.detach().squeeze().numpy().round(3)}")
print(f"Q2 values: {q2.detach().squeeze().numpy().round(3)}")
print(f"Q1-only matches: {torch.allclose(q1, q1_only)}")
print(f"Twin critic params: {sum(p.numel() for p in test_twin.parameters()):,}")

---
## 10. DDPG Agent

In [ ]:
# ============================================================
#  DDPG Agent
# ============================================================

class DDPGAgent:
    """Deep Deterministic Policy Gradient agent.
    
    Components:
        - Actor + target actor (soft-updated)
        - Critic + target critic (soft-updated)
        - Replay buffer
        - Ornstein-Uhlenbeck noise for exploration
    """
    
    def __init__(self, state_dim: int, action_dim: int, max_action: float,
                 hidden_dim: int = HIDDEN_DIM, lr_actor: float = LR_ACTOR,
                 lr_critic: float = LR_CRITIC, gamma: float = GAMMA,
                 tau: float = TAU, buffer_size: int = BUFFER_SIZE,
                 batch_size: int = BATCH_SIZE, seed: int = SEED):
        torch.manual_seed(seed)
        self.action_dim = action_dim
        self.max_action = max_action
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size
        
        # Actor networks
        self.actor = Actor(state_dim, action_dim, hidden_dim, max_action).to(device)
        self.actor_target = copy.deepcopy(self.actor)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        
        # Critic networks
        self.critic = Critic(state_dim, action_dim, hidden_dim).to(device)
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_critic)
        
        # Replay buffer and noise
        self.replay_buffer = ReplayBuffer(buffer_size, seed)
        self.ou_noise = OrnsteinUhlenbeckNoise(action_dim, sigma=0.2, seed=seed)
        
        # Logging
        self.actor_losses = []
        self.critic_losses = []
        self.q_values = []
    
    def select_action(self, state: np.ndarray, noise: bool = True) -> np.ndarray:
        """Select action using the actor, optionally adding OU noise."""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action = self.actor(state_t).cpu().numpy().flatten()
        
        if noise:
            action = action + self.ou_noise()
        
        return np.clip(action, -self.max_action, self.max_action)
    
    def train_step(self) -> Tuple[Optional[float], Optional[float]]:
        """Perform one training step from replay buffer.
        
        Returns:
            (actor_loss, critic_loss) or (None, None) if buffer too small
        """
        if len(self.replay_buffer) < self.batch_size:
            return None, None
        
        # Sample mini-batch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        
        # ---- Critic update ----
        with torch.no_grad():
            # Target actions from target actor
            next_actions = self.actor_target(next_states)
            # Target Q-values
            target_q = self.critic_target(next_states, next_actions)
            # TD target: y = r + gamma * Q'(s', mu'(s')) * (1 - done)
            y = rewards + self.gamma * target_q * (1.0 - dones)
        
        # Current Q-values
        current_q = self.critic(states, actions)
        critic_loss = F.mse_loss(current_q, y)
        
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        # ---- Actor update ----
        # Maximise Q(s, mu(s)) => minimise -Q(s, mu(s))
        actor_loss = -self.critic(states, self.actor(states)).mean()
        
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        
        # ---- Soft target updates ----
        self._soft_update(self.actor, self.actor_target)
        self._soft_update(self.critic, self.critic_target)
        
        # Logging
        self.actor_losses.append(actor_loss.item())
        self.critic_losses.append(critic_loss.item())
        self.q_values.append(current_q.mean().item())
        
        return actor_loss.item(), critic_loss.item()
    
    def _soft_update(self, source: nn.Module, target: nn.Module):
        """Polyak averaging: target <- tau * source + (1 - tau) * target."""
        for src_param, tgt_param in zip(source.parameters(), target.parameters()):
            tgt_param.data.copy_(self.tau * src_param.data + (1.0 - self.tau) * tgt_param.data)


print("DDPGAgent class defined.")

---
## 11. TD3 Agent

In [ ]:
# ============================================================
#  TD3 Agent (Twin Delayed DDPG)
# ============================================================

class TD3Agent:
    """Twin Delayed Deep Deterministic Policy Gradient agent.
    
    Three improvements over DDPG:
        1. Twin critics with min for TD targets (reduce overestimation)
        2. Delayed actor updates (every d steps)
        3. Target policy smoothing (clipped noise on target actions)
    """
    
    def __init__(self, state_dim: int, action_dim: int, max_action: float,
                 hidden_dim: int = HIDDEN_DIM, lr_actor: float = LR_ACTOR,
                 lr_critic: float = LR_CRITIC, gamma: float = GAMMA,
                 tau: float = TAU, buffer_size: int = BUFFER_SIZE,
                 batch_size: int = BATCH_SIZE, policy_delay: int = POLICY_DELAY,
                 policy_noise: float = POLICY_NOISE, noise_clip: float = NOISE_CLIP,
                 exploration_noise: float = EXPLORATION_NOISE, seed: int = SEED):
        torch.manual_seed(seed)
        self.action_dim = action_dim
        self.max_action = max_action
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size
        self.policy_delay = policy_delay
        self.policy_noise = policy_noise
        self.noise_clip = noise_clip
        self.exploration_noise = exploration_noise
        
        # Actor networks
        self.actor = Actor(state_dim, action_dim, hidden_dim, max_action).to(device)
        self.actor_target = copy.deepcopy(self.actor)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        
        # Twin critic networks
        self.critic = TwinCritic(state_dim, action_dim, hidden_dim).to(device)
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_critic)
        
        # Replay buffer (uses Gaussian noise, not OU)
        self.replay_buffer = ReplayBuffer(buffer_size, seed)
        
        # Logging
        self.actor_losses = []
        self.critic_losses = []
        self.q1_values = []
        self.q2_values = []
        self.total_steps = 0
    
    def select_action(self, state: np.ndarray, noise: bool = True) -> np.ndarray:
        """Select action using the actor, optionally adding Gaussian noise."""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action = self.actor(state_t).cpu().numpy().flatten()
        
        if noise:
            action = action + np.random.normal(0, self.exploration_noise, size=self.action_dim)
        
        return np.clip(action, -self.max_action, self.max_action)
    
    def train_step(self, step: int) -> Tuple[Optional[float], Optional[float]]:
        """Perform one training step from replay buffer.
        
        The actor is updated only every `policy_delay` steps.
        
        Args:
            step: current global training step (for delayed updates)
        
        Returns:
            (actor_loss, critic_loss) or (None, critic_loss) on non-actor-update steps
        """
        if len(self.replay_buffer) < self.batch_size:
            return None, None
        
        # Sample mini-batch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        
        # ---- Critic update ----
        with torch.no_grad():
            # Target policy smoothing: add clipped noise to target actions
            noise = (torch.randn_like(actions) * self.policy_noise).clamp(
                -self.noise_clip, self.noise_clip
            )
            next_actions = (self.actor_target(next_states) + noise).clamp(
                -self.max_action, self.max_action
            )
            
            # Twin critic targets: take the minimum
            target_q1, target_q2 = self.critic_target(next_states, next_actions)
            target_q = torch.min(target_q1, target_q2)
            
            # TD target
            y = rewards + self.gamma * target_q * (1.0 - dones)
        
        # Current Q-values from both critics
        current_q1, current_q2 = self.critic(states, actions)
        critic_loss = F.mse_loss(current_q1, y) + F.mse_loss(current_q2, y)
        
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        self.critic_losses.append(critic_loss.item())
        self.q1_values.append(current_q1.mean().item())
        self.q2_values.append(current_q2.mean().item())
        
        # ---- Delayed actor update ----
        actor_loss_val = None
        if step % self.policy_delay == 0:
            # Actor loss: maximise Q1(s, mu(s))
            actor_loss = -self.critic.q1(states, self.actor(states)).mean()
            
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()
            
            actor_loss_val = actor_loss.item()
            self.actor_losses.append(actor_loss_val)
            
            # Soft target updates (only when actor is updated)
            self._soft_update(self.actor, self.actor_target)
            self._soft_update(self.critic, self.critic_target)
        
        return actor_loss_val, critic_loss.item()
    
    def _soft_update(self, source: nn.Module, target: nn.Module):
        """Polyak averaging: target <- tau * source + (1 - tau) * target."""
        for src_param, tgt_param in zip(source.parameters(), target.parameters()):
            tgt_param.data.copy_(self.tau * src_param.data + (1.0 - self.tau) * tgt_param.data)


print("TD3Agent class defined.")

---
## 12. Training Loop

In [ ]:
# ============================================================
#  Unified training function for DDPG and TD3
# ============================================================

def train_agent(
    env_name: str,
    agent,
    n_episodes: int = N_EPISODES,
    max_steps: int = 200,
    warmup_steps: int = WARMUP_STEPS,
    print_every: int = 50,
    seed: int = SEED,
    agent_type: str = 'ddpg',
) -> Dict[str, List[float]]:
    """Train a DDPG or TD3 agent on a Gymnasium environment.
    
    Args:
        env_name:     Gymnasium environment ID
        agent:        DDPGAgent or TD3Agent instance
        n_episodes:   number of training episodes
        max_steps:    max steps per episode
        warmup_steps: random actions before training begins
        print_every:  logging frequency
        seed:         random seed
        agent_type:   'ddpg' or 'td3'
    
    Returns:
        Dictionary with episode rewards and other metrics
    """
    env = gym.make(env_name)
    
    history = {
        'episode_rewards': [],
        'episode_lengths': [],
    }
    
    total_steps = 0
    
    for episode in range(n_episodes):
        state, _ = env.reset(seed=seed + episode)
        episode_reward = 0.0
        
        # Reset OU noise at start of each episode (DDPG only)
        if agent_type == 'ddpg' and hasattr(agent, 'ou_noise'):
            agent.ou_noise.reset()
        
        for step in range(max_steps):
            # Warmup: random actions to fill replay buffer
            if total_steps < warmup_steps:
                action = env.action_space.sample()
            else:
                action = agent.select_action(state, noise=True)
            
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store transition
            agent.replay_buffer.push(state, action, reward, next_state, float(done))
            
            # Train after warmup
            if total_steps >= warmup_steps:
                if agent_type == 'ddpg':
                    agent.train_step()
                else:  # td3
                    agent.train_step(total_steps)
            
            episode_reward += reward
            state = next_state
            total_steps += 1
            
            if done:
                break
        
        history['episode_rewards'].append(episode_reward)
        history['episode_lengths'].append(step + 1)
        
        if (episode + 1) % print_every == 0:
            avg_reward = np.mean(history['episode_rewards'][-50:])
            print(
                f"Episode {episode+1:4d} | "
                f"Reward: {episode_reward:8.2f} | "
                f"Avg(50): {avg_reward:8.2f} | "
                f"Steps: {total_steps:6d}"
            )
    
    env.close()
    return history


def smooth(data: List[float], window: int = 20) -> np.ndarray:
    """Compute running average for smooth plotting."""
    if len(data) < window:
        return np.array(data)
    return np.convolve(data, np.ones(window) / window, mode='valid')


print("Training utilities defined.")

---
## 13. Experiment 1 -- DDPG on Pendulum-v1

In [ ]:
# ============================================================
#  Experiment 1: DDPG on Pendulum-v1
# ============================================================

env_test = gym.make('Pendulum-v1')
state_dim = env_test.observation_space.shape[0]
action_dim = env_test.action_space.shape[0]
max_action = float(env_test.action_space.high[0])
env_test.close()

print(f"Pendulum-v1: state_dim={state_dim}, action_dim={action_dim}, max_action={max_action}")
print(f"Action space: [{env_test.action_space.low[0]}, {env_test.action_space.high[0]}]")
print(f"Observation space: {env_test.observation_space.shape}")

# Create DDPG agent
ddpg_agent = DDPGAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    max_action=max_action,
    seed=SEED,
)

print(f"\nTraining DDPG on Pendulum-v1 ({N_EPISODES} episodes)...")
ddpg_history = train_agent(
    'Pendulum-v1', ddpg_agent,
    n_episodes=N_EPISODES,
    max_steps=200,
    agent_type='ddpg',
)

---
## 14. Experiment 2 -- TD3 on Pendulum-v1

In [ ]:
# ============================================================
#  Experiment 2: TD3 on Pendulum-v1
# ============================================================

# Create TD3 agent
td3_agent = TD3Agent(
    state_dim=state_dim,
    action_dim=action_dim,
    max_action=max_action,
    seed=SEED,
)

print(f"Training TD3 on Pendulum-v1 ({N_EPISODES} episodes)...")
td3_history = train_agent(
    'Pendulum-v1', td3_agent,
    n_episodes=N_EPISODES,
    max_steps=200,
    agent_type='td3',
)

---
## 15. Visualization 1 -- Training Reward Curves: DDPG vs TD3

In [ ]:
# ============================================================
#  Visualization 1: DDPG vs TD3 reward curves (smoothed)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw + smoothed reward curves
ax = axes[0]
window = 20

# DDPG
ax.plot(ddpg_history['episode_rewards'], alpha=0.15, color=COLORS[0])
ddpg_smooth = smooth(ddpg_history['episode_rewards'], window)
ax.plot(np.arange(len(ddpg_smooth)) + window - 1, ddpg_smooth,
        color=COLORS[0], linewidth=2.5, label='DDPG')

# TD3
ax.plot(td3_history['episode_rewards'], alpha=0.15, color=COLORS[1])
td3_smooth = smooth(td3_history['episode_rewards'], window)
ax.plot(np.arange(len(td3_smooth)) + window - 1, td3_smooth,
        color=COLORS[1], linewidth=2.5, label='TD3')

ax.axhline(y=-300, color='black', linestyle='--', alpha=0.4, label='DDPG target (-300)')
ax.axhline(y=-250, color='black', linestyle=':', alpha=0.4, label='TD3 target (-250)')
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward')
ax.set_title('Pendulum-v1: DDPG vs TD3 Training Curves')
ax.legend(loc='lower right')

# Right: rolling standard deviation (stability)
ax = axes[1]
ddpg_std = [np.std(ddpg_history['episode_rewards'][max(0,i-window):i+1])
            for i in range(len(ddpg_history['episode_rewards']))]
td3_std = [np.std(td3_history['episode_rewards'][max(0,i-window):i+1])
           for i in range(len(td3_history['episode_rewards']))]

ax.plot(smooth(ddpg_std, window), color=COLORS[0], label='DDPG', alpha=0.85)
ax.plot(smooth(td3_std, window), color=COLORS[1], label='TD3', alpha=0.85)
ax.set_xlabel('Episode')
ax.set_ylabel('Reward Std Dev (rolling)')
ax.set_title('Training Stability: Reward Variance')
ax.legend()

plt.tight_layout()
plt.show()

---
## 16. Visualization 2 -- Critic Q-Surface

In [ ]:
# ============================================================
#  Visualization 2: 3D Q-value surface over state-action space
# ============================================================

# For Pendulum, state = [cos(theta), sin(theta), theta_dot]
# We fix theta_dot=0 and vary theta and action to visualise Q(s, a)

n_grid = 50
theta_range = np.linspace(-np.pi, np.pi, n_grid)
action_range = np.linspace(-max_action, max_action, n_grid)
theta_grid, action_grid = np.meshgrid(theta_range, action_range)

# Build state tensor: [cos(theta), sin(theta), 0.0]
cos_theta = np.cos(theta_grid).flatten()
sin_theta = np.sin(theta_grid).flatten()
theta_dot = np.zeros_like(cos_theta)
states_grid = torch.FloatTensor(
    np.stack([cos_theta, sin_theta, theta_dot], axis=1)
).to(device)
actions_grid = torch.FloatTensor(action_grid.flatten()).unsqueeze(1).to(device)

# Compute Q-values
with torch.no_grad():
    # DDPG critic
    q_ddpg = ddpg_agent.critic(states_grid, actions_grid).cpu().numpy().reshape(n_grid, n_grid)
    # TD3 critic (Q1)
    q_td3 = td3_agent.critic.q1(states_grid, actions_grid).cpu().numpy().reshape(n_grid, n_grid)

# 3D surface plots
fig = plt.figure(figsize=(16, 6))

ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(theta_grid, action_grid, q_ddpg, cmap='coolwarm', alpha=0.8,
                 edgecolor='none')
ax1.set_xlabel('$\\theta$ (angle)')
ax1.set_ylabel('Action (torque)')
ax1.set_zlabel('Q(s, a)')
ax1.set_title('DDPG: Q-value Surface')
ax1.view_init(elev=25, azim=135)

ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(theta_grid, action_grid, q_td3, cmap='coolwarm', alpha=0.8,
                 edgecolor='none')
ax2.set_xlabel('$\\theta$ (angle)')
ax2.set_ylabel('Action (torque)')
ax2.set_zlabel('Q(s, a)')
ax2.set_title('TD3: Q1-value Surface')
ax2.view_init(elev=25, azim=135)

plt.tight_layout()
plt.show()

---
## 17. Visualization 3 -- Policy Actions Over State Space

In [ ]:
# ============================================================
#  Visualization 3: learned policy mu(s) across the state space
# ============================================================

# Vary theta with theta_dot=0, and also vary theta_dot at theta=0
n_pts = 200
thetas = np.linspace(-np.pi, np.pi, n_pts)
theta_dots = np.linspace(-8, 8, n_pts)

# Policy as function of theta (theta_dot=0)
states_theta = torch.FloatTensor(
    np.stack([np.cos(thetas), np.sin(thetas), np.zeros(n_pts)], axis=1)
).to(device)

# Policy as function of theta_dot (theta=0 => cos=1, sin=0)
states_tdot = torch.FloatTensor(
    np.stack([np.ones(n_pts), np.zeros(n_pts), theta_dots], axis=1)
).to(device)

with torch.no_grad():
    actions_ddpg_theta = ddpg_agent.actor(states_theta).cpu().numpy().flatten()
    actions_td3_theta = td3_agent.actor(states_theta).cpu().numpy().flatten()
    actions_ddpg_tdot = ddpg_agent.actor(states_tdot).cpu().numpy().flatten()
    actions_td3_tdot = td3_agent.actor(states_tdot).cpu().numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Policy vs theta
ax = axes[0]
ax.plot(thetas, actions_ddpg_theta, color=COLORS[0], label='DDPG $\\mu(s)$')
ax.plot(thetas, actions_td3_theta, color=COLORS[1], label='TD3 $\\mu(s)$', linestyle='--')
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('$\\theta$ (angle from upright)')
ax.set_ylabel('Action (torque)')
ax.set_title('Learned Policy vs Angle ($\\dot{\\theta}=0$)')
ax.legend()

# Policy vs theta_dot
ax = axes[1]
ax.plot(theta_dots, actions_ddpg_tdot, color=COLORS[0], label='DDPG $\\mu(s)$')
ax.plot(theta_dots, actions_td3_tdot, color=COLORS[1], label='TD3 $\\mu(s)$', linestyle='--')
ax.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('$\\dot{\\theta}$ (angular velocity)')
ax.set_ylabel('Action (torque)')
ax.set_title('Learned Policy vs Angular Velocity ($\\theta=0$)')
ax.legend()

plt.tight_layout()
plt.show()

---
## 18. Visualization 4 -- Twin Q-Value Divergence

In [ ]:
# ============================================================
#  Visualization 4: TD3 twin Q-value divergence over training
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Q1 and Q2 over training
ax = axes[0]
q1_smooth = smooth(td3_agent.q1_values, window=200)
q2_smooth = smooth(td3_agent.q2_values, window=200)
ax.plot(q1_smooth, color=COLORS[0], label='Q1', alpha=0.85)
ax.plot(q2_smooth, color=COLORS[1], label='Q2', alpha=0.85)
ax.set_xlabel('Training Step')
ax.set_ylabel('Mean Q-value')
ax.set_title('TD3: Twin Q-values Over Training')
ax.legend()

# Q1 - Q2 divergence
ax = axes[1]
q_diff = np.array(td3_agent.q1_values) - np.array(td3_agent.q2_values)
q_diff_smooth = smooth(q_diff.tolist(), window=200)
ax.plot(q_diff_smooth, color=COLORS[4], alpha=0.85)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.4)
ax.fill_between(range(len(q_diff_smooth)), q_diff_smooth, alpha=0.15, color=COLORS[4])
ax.set_xlabel('Training Step')
ax.set_ylabel('Q1 - Q2')
ax.set_title('TD3: Q-value Divergence (Q1 - Q2)')

plt.tight_layout()
plt.show()

# Compare DDPG Q-values vs TD3 (overestimation check)
print(f"DDPG final mean Q: {np.mean(ddpg_agent.q_values[-1000:]):.2f}")
print(f"TD3  final mean Q1: {np.mean(td3_agent.q1_values[-1000:]):.2f}")
print(f"TD3  final mean Q2: {np.mean(td3_agent.q2_values[-1000:]):.2f}")

---
## 19. Visualization 5 -- OU Noise vs Gaussian Noise

In [ ]:
# ============================================================
#  Visualization 5: OU noise vs Gaussian noise comparison
# ============================================================

n_noise_steps = 500

# Generate OU noise trajectory
ou_demo = OrnsteinUhlenbeckNoise(size=1, mu=0.0, theta=0.15, sigma=0.2, seed=42)
ou_traj = [ou_demo()[0] for _ in range(n_noise_steps)]

# Generate Gaussian noise trajectory
np.random.seed(42)
gauss_traj = [np.random.normal(0, 0.2) for _ in range(n_noise_steps)]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time series comparison
ax = axes[0, 0]
ax.plot(ou_traj, color=COLORS[0], alpha=0.8, label='OU Noise')
ax.plot(gauss_traj, color=COLORS[1], alpha=0.5, label='Gaussian Noise')
ax.set_xlabel('Step')
ax.set_ylabel('Noise Value')
ax.set_title('Noise Trajectories: OU vs Gaussian')
ax.legend()

# Autocorrelation
ax = axes[0, 1]
max_lag = 50
ou_autocorr = [np.corrcoef(ou_traj[:-lag], ou_traj[lag:])[0, 1] if lag > 0 else 1.0
               for lag in range(max_lag)]
gauss_autocorr = [np.corrcoef(gauss_traj[:-lag], gauss_traj[lag:])[0, 1] if lag > 0 else 1.0
                  for lag in range(max_lag)]
ax.plot(ou_autocorr, color=COLORS[0], label='OU Noise')
ax.plot(gauss_autocorr, color=COLORS[1], label='Gaussian Noise')
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.set_title('Autocorrelation: OU is Temporally Correlated')
ax.legend()

# Histograms
ax = axes[1, 0]
ax.hist(ou_traj, bins=40, color=COLORS[0], alpha=0.6, label='OU Noise', edgecolor='white')
ax.hist(gauss_traj, bins=40, color=COLORS[1], alpha=0.6, label='Gaussian', edgecolor='white')
ax.set_xlabel('Noise Value')
ax.set_ylabel('Count')
ax.set_title('Noise Distribution')
ax.legend()

# Cumulative effect (integrated noise as "exploration trajectory")
ax = axes[1, 1]
ou_cumulative = np.cumsum(ou_traj)
gauss_cumulative = np.cumsum(gauss_traj)
ax.plot(ou_cumulative, color=COLORS[0], label='OU (cumulative)')
ax.plot(gauss_cumulative, color=COLORS[1], label='Gaussian (cumulative)')
ax.set_xlabel('Step')
ax.set_ylabel('Cumulative Noise')
ax.set_title('Cumulative Noise: OU is Smoother')
ax.legend()

plt.tight_layout()
plt.show()

---
## 20. Visualization 6 -- Actor and Critic Loss Curves

In [ ]:
# ============================================================
#  Visualization 6: Actor and critic loss curves for both agents
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
loss_window = 200

# DDPG Actor Loss
ax = axes[0, 0]
if len(ddpg_agent.actor_losses) > loss_window:
    ax.plot(smooth(ddpg_agent.actor_losses, loss_window), color=COLORS[0], alpha=0.85)
else:
    ax.plot(ddpg_agent.actor_losses, color=COLORS[0], alpha=0.85)
ax.set_xlabel('Training Step')
ax.set_ylabel('Actor Loss')
ax.set_title('DDPG: Actor Loss')

# DDPG Critic Loss
ax = axes[0, 1]
if len(ddpg_agent.critic_losses) > loss_window:
    ax.plot(smooth(ddpg_agent.critic_losses, loss_window), color=COLORS[0], alpha=0.85)
else:
    ax.plot(ddpg_agent.critic_losses, color=COLORS[0], alpha=0.85)
ax.set_xlabel('Training Step')
ax.set_ylabel('Critic Loss')
ax.set_title('DDPG: Critic Loss')

# TD3 Actor Loss
ax = axes[1, 0]
if len(td3_agent.actor_losses) > loss_window:
    ax.plot(smooth(td3_agent.actor_losses, loss_window), color=COLORS[1], alpha=0.85)
else:
    ax.plot(td3_agent.actor_losses, color=COLORS[1], alpha=0.85)
ax.set_xlabel('Training Step')
ax.set_ylabel('Actor Loss')
ax.set_title('TD3: Actor Loss (delayed updates)')

# TD3 Critic Loss
ax = axes[1, 1]
if len(td3_agent.critic_losses) > loss_window:
    ax.plot(smooth(td3_agent.critic_losses, loss_window), color=COLORS[1], alpha=0.85)
else:
    ax.plot(td3_agent.critic_losses, color=COLORS[1], alpha=0.85)
ax.set_xlabel('Training Step')
ax.set_ylabel('Critic Loss')
ax.set_title('TD3: Critic Loss (twin critics)')

fig.suptitle('Actor and Critic Loss Curves: DDPG vs TD3', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 21. Q-Value Overestimation Analysis

In [ ]:
# ============================================================
#  Overestimation analysis: compare predicted Q vs actual returns
# ============================================================

def evaluate_q_accuracy(agent, env_name: str, n_episodes: int = 20,
                        max_steps: int = 200, agent_type: str = 'ddpg',
                        seed: int = SEED) -> Tuple[List[float], List[float]]:
    """Compare predicted Q-values with actual discounted returns.
    
    Returns:
        (predicted_q_values, actual_returns)
    """
    env = gym.make(env_name)
    predicted_qs = []
    actual_returns = []
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + 1000 + ep)
        states_ep = []
        actions_ep = []
        rewards_ep = []
        
        for step in range(max_steps):
            action = agent.select_action(state, noise=False)
            states_ep.append(state.copy())
            actions_ep.append(action.copy())
            
            next_state, reward, terminated, truncated, _ = env.step(action)
            rewards_ep.append(reward)
            state = next_state
            
            if terminated or truncated:
                break
        
        # Compute actual discounted returns for each timestep
        T = len(rewards_ep)
        returns = np.zeros(T)
        G = 0.0
        for t in reversed(range(T)):
            G = rewards_ep[t] + GAMMA * G
            returns[t] = G
        
        # Compute predicted Q-values
        for t in range(T):
            s_t = torch.FloatTensor(states_ep[t]).unsqueeze(0).to(device)
            a_t = torch.FloatTensor(actions_ep[t]).unsqueeze(0).to(device)
            with torch.no_grad():
                if agent_type == 'ddpg':
                    q_pred = agent.critic(s_t, a_t).item()
                else:
                    q_pred = agent.critic.q1(s_t, a_t).item()
            predicted_qs.append(q_pred)
            actual_returns.append(returns[t])
    
    env.close()
    return predicted_qs, actual_returns


# Evaluate both agents
print("Evaluating Q-value accuracy...")
ddpg_pred_q, ddpg_actual_ret = evaluate_q_accuracy(
    ddpg_agent, 'Pendulum-v1', agent_type='ddpg'
)
td3_pred_q, td3_actual_ret = evaluate_q_accuracy(
    td3_agent, 'Pendulum-v1', agent_type='td3'
)

# Scatter plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pred_q, actual_ret, name, color in [
    (axes[0], ddpg_pred_q, ddpg_actual_ret, 'DDPG', COLORS[0]),
    (axes[1], td3_pred_q, td3_actual_ret, 'TD3', COLORS[1]),
]:
    pred_q = np.array(pred_q)
    actual_ret = np.array(actual_ret)
    
    ax.scatter(actual_ret, pred_q, alpha=0.2, s=8, color=color)
    
    lims = [
        min(actual_ret.min(), pred_q.min()) - 20,
        max(actual_ret.max(), pred_q.max()) + 20,
    ]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect prediction')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    
    # Overestimation = mean(predicted - actual)
    overest = np.mean(pred_q - actual_ret)
    corr = np.corrcoef(actual_ret, pred_q)[0, 1] if len(actual_ret) > 1 else 0
    ax.set_xlabel('Actual Return $G_t$')
    ax.set_ylabel('Predicted $Q(s, a)$')
    ax.set_title(f'{name}: Overestimation={overest:.1f}, r={corr:.3f}')
    ax.legend()
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

ddpg_overest = np.mean(np.array(ddpg_pred_q) - np.array(ddpg_actual_ret))
td3_overest = np.mean(np.array(td3_pred_q) - np.array(td3_actual_ret))
print(f"\nDDPG overestimation: {ddpg_overest:.2f}")
print(f"TD3  overestimation: {td3_overest:.2f}")

---
## 22. Soft Target Update Analysis

In [ ]:
# ============================================================
#  Verify soft target updates keep target networks close
# ============================================================

def compute_param_distance(net1: nn.Module, net2: nn.Module) -> float:
    """Compute L2 distance between two networks' parameters."""
    total = 0.0
    for p1, p2 in zip(net1.parameters(), net2.parameters()):
        total += (p1 - p2).pow(2).sum().item()
    return np.sqrt(total)


# Current distances between online and target networks
ddpg_actor_dist = compute_param_distance(ddpg_agent.actor, ddpg_agent.actor_target)
ddpg_critic_dist = compute_param_distance(ddpg_agent.critic, ddpg_agent.critic_target)
td3_actor_dist = compute_param_distance(td3_agent.actor, td3_agent.actor_target)
td3_critic_dist = compute_param_distance(td3_agent.critic, td3_agent.critic_target)

print("Parameter distance between online and target networks:")
print(f"  DDPG Actor:  {ddpg_actor_dist:.6f}")
print(f"  DDPG Critic: {ddpg_critic_dist:.6f}")
print(f"  TD3 Actor:   {td3_actor_dist:.6f}")
print(f"  TD3 Critic:  {td3_critic_dist:.6f}")

# Simulate target tracking over time
print("\nSimulating soft target update tracking...")
track_distances = []
tau_values = [0.001, 0.005, 0.01, 0.05, 0.1]

for tau_test in tau_values:
    # Create a simple test network and target
    test_net = nn.Linear(10, 5)
    test_target = copy.deepcopy(test_net)
    distances = []
    
    for step in range(500):
        # Simulate a parameter update on the online network
        with torch.no_grad():
            for p in test_net.parameters():
                p.add_(torch.randn_like(p) * 0.01)
        
        # Soft update
        for p_src, p_tgt in zip(test_net.parameters(), test_target.parameters()):
            p_tgt.data.copy_(tau_test * p_src.data + (1 - tau_test) * p_tgt.data)
        
        distances.append(compute_param_distance(test_net, test_target))
    
    track_distances.append(distances)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
for i, (tau_test, dists) in enumerate(zip(tau_values, track_distances)):
    ax.plot(dists, color=COLORS[i % len(COLORS)], label=f'$\\tau$={tau_test}', alpha=0.85)

ax.set_xlabel('Update Step')
ax.set_ylabel('L2 Distance (online vs target)')
ax.set_title('Soft Target Update: Parameter Distance for Different $\\tau$ Values')
ax.legend()
plt.tight_layout()
plt.show()

---
## 23. TD3 Ablation: Effect of Each Trick

In [ ]:
# ============================================================
#  Ablation: TD3 with individual tricks disabled
# ============================================================

ablation_configs = {
    'Full TD3': {'policy_delay': 2, 'policy_noise': 0.2, 'noise_clip': 0.5},
    'No delay (d=1)': {'policy_delay': 1, 'policy_noise': 0.2, 'noise_clip': 0.5},
    'No smoothing': {'policy_delay': 2, 'policy_noise': 0.0, 'noise_clip': 0.0},
}

ablation_histories = {}
n_ablation_eps = 200  # shorter for ablation

for name, config in ablation_configs.items():
    print(f"\nTraining {name}...")
    agent_abl = TD3Agent(
        state_dim=state_dim,
        action_dim=action_dim,
        max_action=max_action,
        policy_delay=config['policy_delay'],
        policy_noise=config['policy_noise'],
        noise_clip=config['noise_clip'],
        seed=SEED,
    )
    hist = train_agent(
        'Pendulum-v1', agent_abl,
        n_episodes=n_ablation_eps,
        max_steps=200,
        print_every=100,
        agent_type='td3',
    )
    ablation_histories[name] = hist

# Plot ablation results
fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, hist) in enumerate(ablation_histories.items()):
    smoothed = smooth(hist['episode_rewards'], 15)
    ax.plot(np.arange(len(smoothed)) + 14, smoothed,
            color=COLORS[i], label=name, alpha=0.85)

ax.set_xlabel('Episode')
ax.set_ylabel('Reward (smoothed)')
ax.set_title('TD3 Ablation: Effect of Individual Improvements')
ax.legend()
plt.tight_layout()
plt.show()

---
## 24. Algorithm Summary

In [ ]:
# ============================================================
#  Algorithm comparison table
# ============================================================

summary = """
+================================================================+
|               DDPG vs TD3 Algorithm Comparison                 |
+================================================================+
| Feature              | DDPG                | TD3               |
+================================================================+
| Policy type          | Deterministic       | Deterministic     |
| Critics              | Single Q(s,a)       | Twin Q1, Q2       |
| TD target            | r + g*Q'(s',mu'(s'))| r + g*min(Q1',Q2')|
| Actor update freq    | Every step          | Every d steps     |
| Target smoothing     | No                  | Clipped noise     |
| Exploration          | OU noise            | Gaussian noise    |
| Target update        | Soft (Polyak)       | Soft (Polyak)     |
| Replay buffer        | Yes                 | Yes               |
| Overestimation       | Prone               | Controlled        |
+================================================================+

Hyperparameters Used:
  - GAMMA           = {gamma}
  - TAU             = {tau}
  - LR_ACTOR        = {lr_a}
  - LR_CRITIC       = {lr_c}
  - BATCH_SIZE      = {bs}
  - BUFFER_SIZE     = {buf}
  - POLICY_DELAY    = {pd}  (TD3 only)
  - POLICY_NOISE    = {pn}  (TD3 only)
  - NOISE_CLIP      = {nc}  (TD3 only)
  - EXPLORATION_NOISE = {en}
  - HIDDEN_DIM      = {hd}
  - WARMUP_STEPS    = {ws}
""".format(
    gamma=GAMMA, tau=TAU, lr_a=LR_ACTOR, lr_c=LR_CRITIC,
    bs=BATCH_SIZE, buf=BUFFER_SIZE, pd=POLICY_DELAY,
    pn=POLICY_NOISE, nc=NOISE_CLIP, en=EXPLORATION_NOISE,
    hd=HIDDEN_DIM, ws=WARMUP_STEPS,
)
print(summary)

---
## 25. Final Performance Evaluation

In [ ]:
# ============================================================
#  Evaluate trained agents without noise
# ============================================================

def evaluate_agent(agent, env_name: str, n_episodes: int = 50,
                   max_steps: int = 200, seed: int = SEED) -> List[float]:
    """Evaluate agent without exploration noise."""
    env = gym.make(env_name)
    rewards = []
    
    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + 2000 + ep)
        episode_reward = 0.0
        
        for step in range(max_steps):
            action = agent.select_action(state, noise=False)
            next_state, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward
            state = next_state
            if terminated or truncated:
                break
        
        rewards.append(episode_reward)
    
    env.close()
    return rewards


print("Evaluating trained agents (no noise)...")
ddpg_eval = evaluate_agent(ddpg_agent, 'Pendulum-v1')
td3_eval = evaluate_agent(td3_agent, 'Pendulum-v1')

print(f"\nDDPG: mean={np.mean(ddpg_eval):.1f}, std={np.std(ddpg_eval):.1f}, "
      f"min={np.min(ddpg_eval):.1f}, max={np.max(ddpg_eval):.1f}")
print(f"TD3:  mean={np.mean(td3_eval):.1f}, std={np.std(td3_eval):.1f}, "
      f"min={np.min(td3_eval):.1f}, max={np.max(td3_eval):.1f}")

# Box plot comparison
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([ddpg_eval, td3_eval], labels=['DDPG', 'TD3'],
                patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor(COLORS[0])
bp['boxes'][1].set_facecolor(COLORS[1])
for box in bp['boxes']:
    box.set_alpha(0.7)
ax.set_ylabel('Episode Reward')
ax.set_title('Pendulum-v1: Final Evaluation (50 episodes, no noise)')
plt.tight_layout()
plt.show()

---
## 26. Verification

In [ ]:
# ============================================================
#  Verification Checks
# ============================================================

print("=" * 60)
print("  VERIFICATION CHECKS")
print("=" * 60)

# 1. DDPG achieves avg reward > -300 on Pendulum
ddpg_avg_last50 = np.mean(ddpg_history['episode_rewards'][-50:])
check1 = ddpg_avg_last50 > -300
print(f"\n1. DDPG avg reward > -300 (last 50 training episodes):")
print(f"   Avg reward = {ddpg_avg_last50:.1f}  ->  {'[PASS]' if check1 else '[FAIL]'}")

# 2. TD3 achieves avg reward > -250 on Pendulum
td3_avg_last50 = np.mean(td3_history['episode_rewards'][-50:])
check2 = td3_avg_last50 > -250
print(f"\n2. TD3 avg reward > -250 (last 50 training episodes):")
print(f"   Avg reward = {td3_avg_last50:.1f}  ->  {'[PASS]' if check2 else '[FAIL]'}")

# 3. TD3 has lower Q-value overestimation than DDPG
ddpg_overestimation = np.mean(np.array(ddpg_pred_q) - np.array(ddpg_actual_ret))
td3_overestimation = np.mean(np.array(td3_pred_q) - np.array(td3_actual_ret))
check3 = abs(td3_overestimation) < abs(ddpg_overestimation)
print(f"\n3. TD3 has lower Q-value overestimation than DDPG:")
print(f"   DDPG overestimation: {ddpg_overestimation:.2f}")
print(f"   TD3  overestimation: {td3_overestimation:.2f}")
print(f"   ->  {'[PASS]' if check3 else '[FAIL]'}")

# 4. TD3 training is more stable (lower reward variance in last 50 eps)
ddpg_var = np.std(ddpg_history['episode_rewards'][-50:])
td3_var = np.std(td3_history['episode_rewards'][-50:])
check4 = td3_var < ddpg_var
print(f"\n4. TD3 training is more stable (lower reward std):")
print(f"   DDPG reward std (last 50): {ddpg_var:.2f}")
print(f"   TD3  reward std (last 50): {td3_var:.2f}")
print(f"   ->  {'[PASS]' if check4 else '[FAIL]'}")

# 5. Soft target updates keep target networks close
# Target network params should be close to online network params
max_dist = max(ddpg_actor_dist, ddpg_critic_dist, td3_actor_dist, td3_critic_dist)
check5 = max_dist < 5.0  # reasonable threshold after training
print(f"\n5. Soft target updates keep target networks close:")
print(f"   Max param distance: {max_dist:.4f}")
print(f"   ->  {'[PASS]' if check5 else '[FAIL]'}")

# Overall
checks = [check1, check2, check3, check4, check5]
n_pass = sum(checks)
print(f"\n{'=' * 60}")
print(f"  Overall: {n_pass}/5 checks passed")
print(f"  {'[ALL PASS]' if all(checks) else '[SOME FAILED - results may vary with random seeds]'}")
print(f"{'=' * 60}")

---
## 27. Key Takeaways

1. **Deterministic policies are more sample-efficient** for continuous control because
   the policy gradient does not require integrating over the action space. The DPG
   theorem provides a principled gradient through the Q-function via the chain rule.

2. **DDPG adapts DQN ideas to continuous actions**: experience replay breaks temporal
   correlations, target networks stabilise TD targets, and soft (Polyak) updates
   provide smoother target evolution than hard copies.

3. **OU noise provides temporally correlated exploration**, which is beneficial for
   tasks with momentum/inertia. However, simple Gaussian noise often works just as
   well in practice (as demonstrated by TD3).

4. **TD3 addresses DDPG's key failure mode -- Q-value overestimation** -- with three
   targeted fixes: twin critics (pessimistic Q-targets), delayed policy updates
   (let the critic converge before updating the actor), and target policy smoothing
   (prevent the policy from exploiting narrow Q-function peaks).

5. **All three TD3 tricks complement each other**: the ablation study shows that
   removing any single trick degrades performance, though the twin critics provide
   the largest individual benefit.

**Next steps:** Soft Actor-Critic (SAC), which combines the benefits of off-policy
learning with maximum entropy reinforcement learning for even more robust exploration
and performance.